# exp02: small cross-encoder selector (HotpotQA distractor)

Runtime → Change runtime type → **T4 GPU**. Then *Run all*.

**Results are saved to Google Drive** (`MyDrive/qualiusp2026/exp02_results`) while training runs, so a
disconnect loses at most one checkpoint interval; just re-run all cells and training resumes.
The last cell also downloads a zip of the small result files. Put it in
`experiments/exp02_cross_encoder/results/` in the repo.

Generated by `make_notebook.py`; edit the .py files in the repo, not this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/qualiusp2026/exp02_results'
pathlib.Path(os.environ['RESULTS_DIR']).mkdir(parents=True, exist_ok=True)
pathlib.Path('/content/exp02').mkdir(exist_ok=True)
os.chdir('/content/exp02')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install tiktoken

## Source files (written from the repo)

In [ ]:
%%writefile data.py
"""Data access shared by train / score / analyze. Runs locally and on Colab.

The test (400, = BRACIS paper sample) and dev (1000) questions are fixed by ID in
split_ids.json (exported from exp01), so every experiment sees exactly the same questions.
"""

from __future__ import annotations

import json
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
SPLIT_IDS = json.loads((HERE / "split_ids.json").read_text())


def _load(split: str):
    from datasets import load_dataset

    # Local HF cache has "hotpot_qa"; the hub now also serves it as "hotpotqa/hotpot_qa".
    for name in ("hotpot_qa", "hotpotqa/hotpot_qa"):
        try:
            return load_dataset(name, "distractor", split=split)
        except Exception as e:  # noqa: BLE001
            last = e
    raise last


def load_train(n: int | None = None, seed: int = 0) -> list[dict]:
    ds = _load("train")
    if n:
        ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    return [dict(r) for r in ds]


def load_eval(which: str) -> list[dict]:
    """which in {'test', 'dev'}; returned in split_ids order."""
    ds = _load("validation")
    wanted = SPLIT_IDS[which]
    pos = {i: k for k, i in enumerate(wanted)}
    rows = [None] * len(wanted)
    for r in ds:
        k = pos.get(r["id"])
        if k is not None:
            rows[k] = dict(r)
    missing = sum(r is None for r in rows)
    if missing:
        raise RuntimeError(f"{missing} {which} ids not found in validation split")
    return rows


def paragraphs_of(row: dict) -> tuple[list[str], list[str]]:
    """Paper format: '<title>. <sentences joined>'."""
    titles, paras = [], []
    for title, sents in zip(row["context"]["title"], row["context"]["sentences"], strict=True):
        t = str(title).strip()
        body = " ".join(s.strip() for s in sents if s and s.strip())
        titles.append(t)
        paras.append(f"{t}. {body}" if body else t)
    return titles, paras


def gold_mask(row: dict) -> list[int]:
    gold = {str(t).strip() for t in row["supporting_facts"]["title"]}
    titles, _ = paragraphs_of(row)
    return [int(t in gold) for t in titles]


def build_retrieval_prompt(question: str, paragraphs: list[str]) -> str:
    """Identical to cart-code experiments/preprocessing.py (BRACIS paper)."""
    context_str = "\n\n".join([f"[{i + 1}] {p}" for i, p in enumerate(paragraphs)])
    return f"""Answer the HotpotQA question using only the provided context.

Rules:
- Output only the final answer.
- If the answer is yes, no or noanswer, output exactly: yes or no or noanswer.
- Otherwise output a short span or entity name only.
- Do not include any explanation.
- Do not repeat the question.
- If multiple positions/titles are mentioned, output only the one that
  directly answers the question.

Context:
{context_str}

Question: {question}

Answer:"""


_ENC = None


def n_tokens(text: str) -> int:
    global _ENC
    if _ENC is None:
        import tiktoken

        _ENC = tiktoken.get_encoding("o200k_base")
    return len(_ENC.encode(text))


def results_dir() -> Path:
    """RESULTS_DIR env var (e.g. a Google Drive folder on Colab) or ./results."""
    p = Path(os.environ.get("RESULTS_DIR", HERE / "results"))
    p.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
%%writefile train.py
"""Fine-tune a small cross-encoder to score (question, paragraph) as gold / not gold.

  python train.py --n-train 0   --tag full     # all ~90k train questions
  python train.py --n-train 200 --tag n200 --epochs 10
  python train.py --n-train 50  --tag n50  --epochs 20

Pointwise BCE over all 10 paragraphs of each question (2 gold, 8 BM25 distractors).
Checkpoints are saved to RESULTS_DIR/<tag>/ every --save-every steps and resumed
automatically, so a Colab disconnect only loses the current interval.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import time

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from data import gold_mask, load_train, paragraphs_of, results_dir

BASE = "cross-encoder/ms-marco-MiniLM-L6-v2"


def pairs_from(rows):
    out = []
    for r in rows:
        _, paras = paragraphs_of(r)
        for p, y in zip(paras, gold_mask(r), strict=True):
            out.append((r["question"], p, float(y)))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n-train", type=int, default=0, help="0 = all train questions")
    ap.add_argument("--tag", required=True)
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--max-len", type=int, default=256)
    ap.add_argument("--save-every", type=int, default=2000)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    random.seed(args.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    out = results_dir() / args.tag
    out.mkdir(parents=True, exist_ok=True)
    ckpt = out / "checkpoint.pt"

    rows = load_train(args.n_train or None, seed=args.seed)
    pairs = pairs_from(rows)
    print(f"train questions={len(rows)} pairs={len(pairs)} device={dev}")

    tok = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(BASE).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
    steps_per_epoch = math.ceil(len(pairs) / args.batch)
    total = steps_per_epoch * args.epochs
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (total - s) / max(1, total - warm))
    )
    scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda")

    step = 0
    if ckpt.exists():
        st = torch.load(ckpt, map_location=dev)
        model.load_state_dict(st["model"])
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        step = st["step"]
        print(f"resumed from step {step}/{total}")

    def save():
        torch.save(
            {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
             "step": step}, ckpt)

    model.train()
    t0 = time.time()
    while step < total:
        epoch = step // steps_per_epoch
        order = list(range(len(pairs)))
        random.Random(args.seed + epoch).shuffle(order)
        for b in range(step % steps_per_epoch, steps_per_epoch):
            idx = order[b * args.batch : (b + 1) * args.batch]
            q, p, y = zip(*(pairs[i] for i in idx), strict=True)
            enc = tok(list(q), list(p), truncation=True, max_length=args.max_len,
                      padding=True, return_tensors="pt").to(dev)
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=dev == "cuda"):
                logits = model(**enc).logits.squeeze(-1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits.float(), torch.tensor(y, device=dev))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 100 == 0:
                rate = step / (time.time() - t0)
                print(f"step {step}/{total} loss {loss.item():.4f} ({rate:.1f} steps/s)", flush=True)
            if step % args.save_every == 0:
                save()
            if step >= total:
                break

    model.save_pretrained(out / "model")
    tok.save_pretrained(out / "model")
    (out / "train_args.json").write_text(json.dumps(
        {**vars(args), "base": BASE, "n_questions": len(rows), "n_pairs": len(pairs),
         "steps": total}, indent=1))
    if ckpt.exists():
        ckpt.unlink()  # final model saved; drop optimizer state to save Drive space
    print(f"saved {out / 'model'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile score.py
"""Score every (question, paragraph) of the dev and test questions with a cross-encoder.

  python score.py --model zeroshot          # off-the-shelf ms-marco MiniLM (arm A0)
  python score.py --model full              # RESULTS_DIR/full/model

Writes RESULTS_DIR/scores_<name>.json : {split: {qid: [10 logits]}}. Small (~0.5 MB),
everything else (selection, tokens, recall) is computed from it by analyze.py.
"""

from __future__ import annotations

import argparse
import json

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from data import load_eval, paragraphs_of, results_dir
from train import BASE


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="'zeroshot' or a tag under RESULTS_DIR")
    ap.add_argument("--max-len", type=int, default=256)
    ap.add_argument("--batch", type=int, default=64)
    args = ap.parse_args()

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    path = BASE if args.model == "zeroshot" else str(results_dir() / args.model / "model")
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(dev).eval()

    out = {}
    for split in ("dev", "test"):
        rows = load_eval(split)
        pairs, owner = [], []
        for r in rows:
            _, paras = paragraphs_of(r)
            for p in paras:
                pairs.append((r["question"], p))
                owner.append(r["id"])
        logits = []
        with torch.no_grad():
            for i in range(0, len(pairs), args.batch):
                q, p = zip(*pairs[i : i + args.batch], strict=True)
                enc = tok(list(q), list(p), truncation=True, max_length=args.max_len,
                          padding=True, return_tensors="pt").to(dev)
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=dev == "cuda"):
                    logits += model(**enc).logits.squeeze(-1).float().cpu().tolist()
        per_q: dict[str, list[float]] = {}
        for qid, s in zip(owner, logits, strict=True):
            per_q.setdefault(qid, []).append(s)
        out[split] = per_q
        print(f"{split}: scored {len(pairs)} pairs")

    dest = results_dir() / f"scores_{args.model}.json"
    dest.write_text(json.dumps(out))
    print(f"wrote {dest}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze.py
"""Selection metrics from score files (no GPU, no LLM). Same protocol as exp01.

  python analyze.py                      # every results/scores_*.json

For each scorer:
  - top-k (k=1..5): both-gold rate ("full recall"), precision, tokens
  - adaptive: keep paragraphs with sigmoid(score) >= t (at least 1); t chosen on DEV
    for full-recall targets 0.90 / 0.95, then reported once on TEST.
Tokens = the paper's exact prompt, tiktoken o200k_base.
"""

from __future__ import annotations

import json

import numpy as np
import pandas as pd

from data import build_retrieval_prompt, gold_mask, load_eval, n_tokens, paragraphs_of, results_dir


def prep(split):
    items = {}
    for r in load_eval(split):
        _, paras = paragraphs_of(r)
        items[r["id"]] = {
            "gold": np.array(gold_mask(r), bool),
            "base": n_tokens(build_retrieval_prompt(r["question"], [])),
            "ptok": np.array([n_tokens(f"[{i + 1}] {p}\n\n") for i, p in enumerate(paras)]),
        }
    return items


def metrics(items, scores, select):
    full, prec, npar, tok = [], [], [], []
    for qid, it in items.items():
        s = np.asarray(scores[qid])
        sel = select(s)
        hit = it["gold"][sel].sum()
        full.append(hit == it["gold"].sum())
        prec.append(hit / len(sel))
        npar.append(len(sel))
        tok.append(it["base"] + it["ptok"][sel].sum())
    return {"full_recall": np.mean(full), "precision": np.mean(prec),
            "n_paras": np.mean(npar), "tokens": np.mean(tok)}


def topk(k):
    return lambda s: np.argsort(-s)[:k]


def thresh(t):
    def f(s):
        p = 1 / (1 + np.exp(-s))
        keep = np.where(p >= t)[0]
        return keep[np.argsort(-s[keep])] if len(keep) else np.argsort(-s)[:1]
    return f


def main():
    rdir = results_dir()
    files = sorted(rdir.glob("scores_*.json"))
    if not files:
        raise SystemExit(f"no scores_*.json in {rdir}")
    items = {"dev": prep("dev"), "test": prep("test")}
    rows = []
    for f in files:
        name = f.stem.removeprefix("scores_")
        sc = json.loads(f.read_text())
        for k in range(1, 6):
            rows.append({"scorer": name, "rule": f"top{k}",
                         **{f"test_{m}": v for m, v in metrics(items["test"], sc["test"], topk(k)).items()}})
        grid = np.round(np.concatenate([
            [1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3],
            np.arange(0.01, 0.1, 0.01), np.arange(0.1, 0.96, 0.025)]), 4)
        dev = pd.DataFrame([{"t": t, **metrics(items["dev"], sc["dev"], thresh(t))} for t in grid])
        dev.to_csv(rdir / f"dev_thresholds_{name}.csv", index=False)
        for target in (0.85, 0.90, 0.95):
            ok = dev[dev.full_recall >= target]
            if not len(ok):
                continue
            t = float(ok.sort_values("tokens").iloc[0].t)
            rows.append({"scorer": name, "rule": f"thresh@{target} (t={t})",
                         **{f"test_{m}": v for m, v in metrics(items["test"], sc["test"], thresh(t)).items()}})
    df = pd.DataFrame(rows)
    df.to_csv(rdir / "summary.csv", index=False)
    pd.set_option("display.width", 200)
    print(df.round(3).to_string(index=False))
    print("\nReference (exp01, test): k=10 1442 tok; Noise-Gate tau=0.20 0.928/1221 tok;"
          " hop-gate 0.918/1089 tok; gold-only 323 tok")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile split_ids.json
{"test": ["5ae143ed55429920d5234360", "5abc19705542993a06baf86e", "5ac3e0f7554299194317388b", "5ae518655542993aec5ec139", "5ab985eb554299131ca42360", "5a8753d95542994846c1cd63", "5ab97d0a5542996be202051e", "5adef1b35542993a75d263af", "5a73b55855429978a71e9086", "5ae80919554299540e5a56f6", "5ae361605542994393b9e69b", "5ac3a76e554299741d48a2be", "5a7557d75542992d0ec05f68", "5a8d12ca5542994ba4e3dbe2", "5ab7f6d35542993667794070", "5ac3d9135542995c82c4ac4c", "5ae129115542990adbacf722", "5a86769c5542994775f60776", "5ab31864554299233954ff06", "5a7b1023554299042af8f6c2", "5ac19f405542991316484b5b", "5ae382ee5542992e3233c430", "5adcdf455542994ed6169c11", "5ac35a7e554299741d48a257", "5ab2659e554299340b5254b2", "5adcbb815542994ed6169bde", "5a74f8bd5542993748c8976a", "5ade25ed5542997c77aded70", "5a78ae8b5542990784727730", "5ae12a4b55429920d52342df", "5ae22eae554299495565da30", "5ab6aa4c55429953192ad359", "5adf3e835542993a75d26442", "5a8ee3a755429917b4a5be02", "5ac0dcf25542996f0d89cc2c", "5a7e618155429949594199b0", "5ab916e155429919ba4e23a4", "5a713b325542994082a3e6b5", "5a87c13f5542996e4f30890c", "5a8843cd5542994846c1ce75", "5ab90d8955429916710eb0f1", "5ae788fa55429952e35ea964", "5ae1925a554299492dc91b48", "5ae1443855429920d5234362", "5ac54c435542993e66e822d1", "5a7a77425542995eb53be83e", "5a7153d05542994082a3e7dc", "5ae361805542992e3233c3c9", "5a72688c5542997f827839b2", "5a7ce0ce554299452d57ba92", "5a83411655429966c78a6b5d", "5a877dd65542993e715abf79", "5ab6087a554299110f2199be", "5adbd70c55429947ff173843", "5a7796e05542992a6e59df0f", "5abc030e554299642a094bdc", "5a7bb7f2554299042af8f7b6", "5a8b3de95542997f31a41cc3", "5ab3bfd3554299233954ff99", "5ab9b7d555429970cfb8eb7a", "5ab6bf6d55429953192ad372", "5ae7cb9b5542994a481bbde2", "5ab1d6545542997061209588", "5addcb215542997dc7907033", "5ae528ed5542993aec5ec16e", "5a8c7ced5542995e66a4761c", "5a72f74a55429901807daf59", "5ae1d4e8554299234fd04312", "5a80aeef5542992097ad301a", "5a7f66f155429969796c1a33", "5abbda84554299642a094b5b", "5abe3dc65542993f32c2a0c4", "5ae489ab5542995ad6573d62", "5a7e1b7155429965cec5ea6c", "5a8fae205542997ba9cb32ec", "5ae588cb55429960a22e0300", "5a75e62f5542992d0ec06013", "5ac113e2554299294b21909a", "5a82fe5055429966c78a6ad3", "5a7e05ba5542995f4f402392", "5a7264ed5542990c210a411a", "5a8ed10b5542995085b374a0", "5add8f055542990dbb2f7e6d", "5a86ab7755429960ec39b6bf", "5a7a112a5542990783324e2b", "5ab870ee5542990e739ec8f6", "5a8810485542997e5c09a590", "5a8208ee5542990a1d231f1e", "5a86399e5542994775f60733", "5a78e9a2554299148911f995", "5a84995d5542997b5ce3fee4", "5ab941c2554299743d22ea88", "5a8e5f1f5542995a26add4d6", "5abf1ad15542997ec76fd3c2", "5ae5cf625542996de7b71a22", "5a7c8ae855429935c91b5207", "5a87106555429960ec39b71f", "5a7f296f55429934daa2fd10", "5ae14b5c55429920d52343aa", "5a88b3b4554299206df2b336", "5add67705542992ae4cec54f", "5a875fee5542996e4f3087ab", "5ab2b630554299545a2cfa17", "5a851ba95542997175ce1f81", "5ae1d0e8554299234fd0430a", "5ae1ebf95542997283cd22f2", "5a81d0515542990a1d231ed4", "5ae35e8c5542990afbd1e120", "5a87ed5b5542994846c1cdfc", "5abc41315542993a06baf8bb", "5a7be1ab5542997c3ec972a9", "5a840e395542992ef85e239d", "5ae8167b55429952e35eaa06", "5a776a805542993735360269", "5ab4b9f75542990594ba9ca4", "5a78e3b955429974737f78f1", "5a851b0d5542997175ce1f7f", "5a78b5f7554299148911f95b", "5ab4304a55429942dd415ec5", "5a72dcb45542992359bc31af", "5a7e0ed25542991319bc9424", "5ab9e477554299232ef4a25d", "5ab900045542991b5579f0c0", "5ab6e848554299710c8d1faa", "5a7dc2495542997cc2c4749a", "5a84e9d35542994c784dda84", "5a7cfea2554299452d57babe", "5ae497f15542995ad6573db8", "5ae2133e554299495565d9e3", "5abfe1115542997d64295909", "5a8ce55f554299441c6b9f62", "5a7c47c35542990527d55481", "5ab4107a554299753aec5a2f", "5a774e415542994aec3b7280", "5abdf06355429976d4830a2d", "5a8ac4dd5542992d82986f76", "5a8f503c5542992414482a34", "5a8ec55f5542995a26add50c", "5ae1338d55429901ffe4ade1", "5a7bea3d5542996dd594b847", "5a7f567e5542992097ad2f22", "5ac26ac15542992f1f2b38bc", "5adbcc085542996e6852523c", "5ae531695542990ba0bbb1fb", "5ac2cb09554299657fa29066", "5a82817555429954d2e2eb5a", "5ae34b275542992e3233c371", "5a74f5155542993748c89750", "5add0f635542994ed6169c56", "5ab4428c5542990594ba9bda", "5a721a7655429971e9dc9271", "5ae532365542992663a4f150", "5a72f1cf5542994cef4bc414", "5a749e1855429979e28829c3", "5a8b051255429971feec460e", "5a7616755542992d0ec06028", "5ac0fd56554299012d1db679", "5adde73f5542992200553b94", "5ab506a05542990594ba9cd6", "5a7f57a45542992097ad2f29", "5a8dee2455429917b4a5bce1", "5a7cd7f055429909bec7683e", "5a7cb8bc554299452d57b9be", "5ab9547755429970cfb8ea6c", "5a8cfa2e554299585d9e378b", "5ab67bd355429954757d32f2", "5abecdf45542997719eab5d4", "5a7e37095542995ed0d166d5", "5a8f18d555429918e830d189", "5a7b85ac5542997c3ec971ef", "5ae617285542996de7b71b4c", "5ae7247b5542992ae0d163c1", "5ae0ae4555429945ae959419", "5ae306115542992decbdcdc4", "5ade4fcc5542997c77adeded", "5abdd08d5542991f6610604c", "5a72abb95542992359bc3162", "5a7f54665542992097ad2f1a", "5ae0bc185542993d6555ec29", "5a864d9e5542994775f6073d", "5ae5b25d5542990ba0bbb2bb", "5ae5e62455429929b0807a07", "5ae0120155429925eb1afbfb", "5abe87cb5542993f32c2a156", "5ae2b770554299495565db0f", "5a7af2a455429931da12c998", "5a7af23a55429931da12c994", "5ab8634e5542990e739ec8de", "5a8836445542997e5c09a5a5", "5ae127c9554299422ee99612", "5ac194db55429964131be23a", "5ab8eabb5542991b5579f03d", "5ab234c65542993be8fa98c6", "5ae524415542992663a4f121", "5a7e42df5542994959419954", "5abffc0d5542990832d3a1e2", "5ae6914755429908198fa627", "5ae78f3b554299540e5a5608", "5a8b46925542995d1e6f133c", "5ab4f7fc5542991779162d43", "5abd13b655429924427fceea", "5adf37a95542995ec70e8f97", "5a78e86f55429974737f78fb", "5a807ceb554299485f598616", "5a8c493e554299653c1aa020", "5a75404a55429916b01642be", "5a81108855429926c1cdace2", "5ae340a85542994393b9e668", "5a8d0be85542994ba4e3dbb2", "5add43335542995b365faae1", "5ae4d8475542993aec5ec0e0", "5ab5c263554299488d4d9a18", "5a74a26a55429979e28829d6", "5a8d841955429941ae14dfe6", "5a809e9f5542996402f6a5b1", "5ac2d85e55429921a00ab06b", "5a8319b255429940e5e1a960", "5a7aa0a55542990198eaf165", "5ab9ddef5542994dbf019875", "5ae157ed5542990adbacf75d", "5a8555ea5542992a431d1b07", "5ae141a65542997b2ef7d18a", "5ae4a1ef55429970de88d9e7", "5ab90e9755429934fafe6e51", "5a8e3ea95542995a26add48d", "5ac2f34455429921a00ab0b6", "5a7344e95542991f9a20c6ce", "5a7b1c6b55429931da12c9ca", "5ab341f755429969a97a8114", "5a8b50475542997f31a41cd4", "5a8cddd1554299441c6b9f3a", "5a88f7855542993b751ca8da", "5ab90d2755429916710eb0f0", "5a7300b85542994cef4bc41a", "5ae7f56e5542993210983fd1", "5a7562045542992db947364f", "5adda7585542997dc790700c", "5ab4314955429942dd415ecd", "5a81a2705542990a1d231e30", "5ae291bb5542996483e649f6", "5ab207f65542993be8fa98b8", "5ae152a65542990adbacf74f", "5a739ef955429978a71e904b", "5ab2b61d5542991669774065", "5a88fc4255429938390d4019", "5ae2c416554299492dc91c67", "5a7a51b95542990783324f0f", "5ae3006c55429928c423959f", "5a74122f55429929fddd83da", "5a81711455429938b614233e", "5ae1e4095542997283cd22bf", "5a8c6d4e5542996e8ac88aa6", "5abf11d45542997719eab660", "5ac11201554299294b21908c", "5ade063d5542995b365fabcd", "5ab3a83d5542992ade7c6e08", "5a79b525554299029c4b5f67", "5a8b77705542995d1e6f13aa", "5a8aeadc55429950cd6afbe0", "5abd0296554299114383a1d1", "5ae4d0ff55429913cc204511", "5ac07fff554299294b219006", "5ab741005542992aa3b8c800", "5abe4a365542993f32c2a0dc", "5ab6cebd55429954757d3378", "5abc219655429959677d6a1d", "5ac546975542993e66e8229a", "5adf4da15542995534e8c780", "5a7f1d3c5542993067513639", "5ae3c71f5542991a06ce9a22", "5a7e32905542991319bc943b", "5a75462d5542996c70cfaed3", "5a8effb55542997ba9cb317f", "5abec66b5542997ec76fd360", "5a9030ef5542995651fb50e4", "5a762cf0554299109176e6a4", "5ade1c9e55429939a52fe822", "5abb1e1b5542992ccd8e7f19", "5a791ce955429974737f798e", "5ae67dba55429908198fa5f0", "5a7ef6b35542994959419a94", "5ae5e8dc554299546bf82fbe", "5a89bbb05542992e4fca83a3", "5adcf8155542990d50227d6a", "5a74feb75542996c70cfae6d", "5ae32e125542991a06ce9946", "5a8242c155429940e5e1a81d", "5ab2acbc55429929539467d2", "5abd7d0c55429924427fd010", "5ae6860e5542991bbc976112", "5a849cc95542997175ce1efa", "5ac1a04e5542991316484b63", "5ade7a435542997c77adee3a", "5a865e8a55429960ec39b67a", "5adc0e0355429944faac2404", "5a80ecea55429938b6142254", "5ae7cea355429952e35ea9c1", "5a77a44d5542997042120aba", "5a775f8a5542994aec3b72ba", "5a80870d5542992bc0c4a752", "5ae4d7fc5542990ba0bbb16f", "5abe765a55429976d4830b1f", "5a84f5195542994c784dda9d", "5ab431e65542991751b4d6e2", "5ae71f06554299572ea54708", "5abdce1b5542993062266cfe", "5a8c5eca554299240d9c213d", "5a8c8f24554299653c1aa0ba", "5ab972485542996be20204f6", "5a778a8e5542992a6e59deca", "5ab285d1554299449642c8dd", "5a78e49855429974737f78f3", "5a72b3335542994cef4bc3dc", "5a82bc7455429940e5e1a926", "5a77b0395542997042120ae1", "5a90b6985542990a984936c2", "5a8a833855429970aeb702fe", "5a7fd95d5542992e7d278dc5", "5ab1e36d554299449642c7f9", "5adfff0755429925eb1afbce", "5ab5e024554299494045f097", "5abfa2255542993fe9a41e18", "5a810506554299260e20a202", "5a71231a5542994082a3e5c4", "5a71717d5542994082a3e834", "5ab707d65542991d32223761", "5a74adb455429916b01641c6", "5abe34095542993f32c2a088", "5ab28c89554299340b525513", "5ab626d555429953192ad279", "5ab3589455429976abd1bc93", "5a7cc59c55429909bec767fb", "5ab827a555429934fafe6cf5", "5a8af1e255429950cd6afc28", "5ac11ee45542997d64295a7d", "5ae520315542993aec5ec152", "5a8f68795542992414482ab5", "5ae37e525542992e3233c42d", "5ab849d955429916710eb017", "5ade51515542992fa25da745", "5ade02d05542997545bbbe0d", "5ac41850554299194317390b", "5a727e1b5542991f9a20c497", "5a8d9de2554299068b959d62", "5a827ce155429940e5e1a8d7", "5ae5415055429908b63265b7", "5ab5ec0a5542997d4ad1f250", "5ae20427554299234fd04397", "5adcd7325542994d58a2f6f6", "5abace6b5542996cc5e49e99", "5ab64cfa55429953192ad2b7", "5ae4bb1255429913cc2044e7", "5abe2b0155429976d4830a83", "5ae0d52e554299603e418469", "5abc9fb055429959677d6ac9", "5ac083b0554299294b21900d", "5a8b3e985542995d1e6f131c", "5a75f0ea5542994ccc91866c", "5a7c6b6655429935c91b519a", "5ab295eb554299194fa93412", "5a7137dc5542994082a3e68a", "5a82d71555429966c78a6a94", "5abb7ea4554299642a094a96", "5a7f00b455429930675135f1", "5ae153915542990adbacf752", "5a7bf3da5542996dd594b869", "5abb4b4a5542993f40c73aea", "5ae1f2f3554299234fd0435a", "5a8f0796554299458435d528", "5ae81b895542997ec272770c", "5ae0f6f5554299422ee99583", "5a7c6d98554299683c1c6304", "5a83de04554299334474609f", "5ab29c24554299449642c932", "5ae20c245542997283cd2367", "5ac4efa85542991943173a04", "5ac385f8554299741d48a279", "5ab4355555429942dd415ee3", "5add23d05542994734353871", "5ab99a215542996be202056a", "5add15d15542994ed6169c70", "5ae5d82f554299546bf82f9a", "5ab29b07554299194fa93440", "5a88bedb554299206df2b35a", "5abfd3ba5542994516f4550a", "5a728b1c5542992359bc30e0", "5a806aca5542996402f6a503", "5ae4bbf05542995ad6573e32", "5a8638945542994775f6072d", "5ab3bcc55542992ade7c6e66", "5ab7b0775542992aa3b8c83e", "5abd68b45542993062266c77", "5ac2a399554299218029dada", "5a8f2fed55429918e830d1c7", "5a80abb65542996402f6a5df", "5a7fc8d05542992e7d278d86", "5a8f45ca5542997ba9cb320e", "5a87aa135542996e4f3088b8", "5ab29346554299545a2cf997", "5ae789615542997ec2727695"], "dev": ["5a74c3f455429979e2882a0a", "5aba9e545542994dbf019986", "5a7b59115542995eb53be923", "5abd0eae5542996e802b46d4", "5ab83d595542990e739ec886", "5add98985542990dbb2f7e82", "5ab514dc5542990594ba9cfa", "5abc6d14554299700f9d78bf", "5a7f3f7c55429934daa2fd45", "5ae361dc5542992f92d822aa", "5adc9ebc5542994d58a2f681", "5ab58f9a554299488d4d99da", "5abc0aba55429959677d6a00", "5adf2fa35542993344016c11", "5a7c6bcc55429935c91b519c", "5ae7c5ef5542993210983f44", "5ab3acc5554299233954ff7e", "5adcf20f5542994ed6169c2c", "5adfdc7d55429942ec259b64", "5ac2a3c7554299218029dadb", "5a7af4cd55429931da12c9a6", "5a73024f5542991f9a20c5fc", "5a86ec9f55429960ec39b6d7", "5ac19d4c55429963665198bb", "5a870a8e55429960ec39b707", "5ab9631a55429970cfb8eaad", "5ac270025542990b17b15390", "5adf39765542993a75d26435", "5ae8027d5542993210984031", "5a802b595542995d8a8ddf51", "5add750c5542990dbb2f7e53", "5ab8f3235542991b5579f084", "5a90b1635542990a984936a9", "5ae22a08554299234fd0440b", "5a815f445542995ce29dcbec", "5add564a5542990dbb2f7e38", "5a7db62d5542997cc2c4747c", "5ae32b6755429928c4239644", "5a8ba3ff55429971feec4744", "5a8e08d35542995085b37395", "5ab3f7b05542992ade7c6efd", "5a7d2b535542995f4f402224", "5a8b448255429971feec468c", "5abbbaca55429931dba144ca", "5a8b017155429949d91db463", "5a843de75542992ef85e23b4", "5ac2eec55542990b17b154c1", "5ae1f8235542997f29b3c1cf", "5a7e689f55429934daa2fc1c", "5a77101d5542994aec3b71d7", "5a8d5f16554299585d9e37c5", "5a77ddb855429967ab1052d8", "5ae13c5155429920d5234336", "5ac2455e55429951e9e68512", "5a8595005542992a431d1b40", "5ab64285554299637185c67c", "5ae655845542991bbc9760c3", "5a88bd1d554299206df2b357", "5ae7f6be554299540e5a56db", "5a820cb355429903bc27ba2f", "5a7e1ad155429965cec5ea66", "5abdbf3755429933744ab818", "5ade9c9c5542997c77adee8c", "5adfa73b55429942ec259ad8", "5a7da3a95542990b8f5039e6", "5a7f251155429934daa2fcf2", "5a8483b45542992a431d1a40", "5a84ee4c5542997175ce1f2b", "5a74be7a55429929fddd84e4", "5ab430f655429942dd415eca", "5adf82945542993344016ca4", "5a79dae35542994f819ef0d3", "5a73520c5542994cef4bc53f", "5a907bf05542995651fb5164", "5a776bc855429966f1a36d3a", "5adcf83855429947343537e1", "5ae641f65542992ae0d162a3", "5ac5484d5542994611c8b48d", "5a7f8d755542992097ad2f9d", "5abea85f5542990832d3a070", "5abe18e055429976d4830a74", "5ae4808455429970de88d990", "5ab8829d55429934fafe6e08", "5adcdc6c5542992c1e3a2437", "5a714f6f5542994082a3e7b3", "5a8e17ce554299653c1aa169", "5ae0f5fc554299422ee9957c", "5ae8295955429952e35eaa6d", "5adc2f53554299438c868d6d", "5abbfb5e554299114383a0d0", "5ac39df8554299741d48a2b6", "5a806cdf5542996402f6a508", "5ae803915542994a481bbe80", "5ab294a6554299545a2cf9a6", "5a77b5c255429967ab10523c", "5a7299465542992359bc3131", "5a7e4af3554299495941998c", "5ae835f25542997ec272776f", "5abb8ebe5542993f40c73b2d", "5adfa22655429942ec259ac4", "5ab5f854554299637185c64e", "5a8f51695542992414482a3e", "5ab6a8b5554299710c8d1f30", "5a7724715542994aec3b71fd", "5a7619765542992db9473728", "5a8849ac5542996e4f308932", "5ae724ec5542995703ce8bd8", "5a77767a55429967ab10519b", "5a78d35e55429974737f78c3", "5a81ea115542995ce29dcc78", "5abc546b554299700f9d786c", "5ae81060554299540e5a572c", "5a8b71915542995d1e6f1393", "5ae08dad55429945ae9593b9", "5ae760415542997b22f6a708", "5addfe1c5542995b365fabaf", "5ade1eda55429975fa854e71", "5adfad2b554299603e41835c", "5ae141475542997b2ef7d185", "5a83041355429966c78a6ae8", "5ae5fb55554299546bf82ff6", "5a7143625542994082a3e71e", "5a745a6055429974ef308be4", "5ae56fb85542993aec5ec1a9", "5a901e5c55429933b8a2047c", "5a7c53a855429935c91b515d", "5ac3e68b5542997ea680c98b", "5ab7f1b65542991d322237d3", "5a89382655429951533612e8", "5a7528e955429929fddd850e", "5a8f45385542997ba9cb320a", "5ab7ff1d5542990e739ec7b9", "5a7d2c8d5542995ed0d16624", "5a8510a25542994c784ddaf0", "5ae2e61455429928c423953e", "5ab680d15542995eadef0011", "5abed44b5542990832d3a0db", "5a8a5dd055429970aeb702c7", "5abcf299554299700f9d7936", "5a87bada5542994846c1cddc", "5a807e6f5542995d8a8ddf80", "5a750d315542996c70cfae84", "5a8762205542996e4f3087b8", "5a7e23655542997cc2c47528", "5a73c958554299623ed4ac5c", "5a8ed30655429917b4a5bdcd", "5ae650fb55429929b0807b2e", "5ab6bb575542995eadef0088", "5a8a4b9955429930ff3c0d88", "5ab9c7c355429939ce03dc17", "5ae020f5554299025d62a41b", "5ac3caa15542995ef918c23a", "5abe044d5542991f661060a7", "5a820f2f554299676cceb1ff", "5a7ba83f5542995eb53be981", "5ae13e205542997b2ef7d157", "5ae1436e55429901ffe4ae31", "5ae0819155429924de1b70ea", "5ac005a15542997719eab6fc", "5ade8ad95542992fa25da7b8", "5a8e341c5542995085b373d6", "5a76364d5542994ccc918719", "5a732aa055429901807daf9f", "5abeab375542990832d3a091", "5abee4ff5542993fe9a41d7d", "5ac175635542994ab5c67d4b", "5ab797af5542995dae37e96c", "5ac313145542995ef918c0ed", "5addc7e35542997545bbbdbe", "5adcb6875542994ed6169bcb", "5ac31602554299741d48a1bf", "5ac05fb55542997d64295a0b", "5a72a3c45542994cef4bc3ac", "5a8841195542994846c1ce65", "5ae10a5b55429920d52342a0", "5a7bbcd65542997c3ec97268", "5a8b595855429949d91db563", "5ab7f3285542992aa3b8c88f", "5ab53436554299637185c504", "5a778cae55429949eeb29eeb", "5abc8d75554299700f9d7900", "5ab4eecb55429942dd415faa", "5a77d2d25542992a6e59dfe2", "5ae23d945542996483e64939", "5ae7ed9555429952e35ea9e6", "5ae692e35542996d980e7c0e", "5ac007cc5542996f0d89cb0b", "5a8ef2a75542995a26add583", "5ae75a5a5542991bbc9761f3", "5ab677e2554299110f219a13", "5a835a265542992ef85e2281", "5ab4482b55429942dd415f09", "5a84564c5542990548d0b288", "5a82626655429966c78a6a08", "5ae0d7bf5542990adbacf67b", "5a7119f75542994082a3e58e", "5a7a06485542990198eaf04e", "5ae70c7b5542991bbc9761aa", "5ae73c00554299572ea5478a", "5a7e355055429934daa2fbf4", "5ab9988b554299131ca4238e", "5ab414295542996a3a969f1d", "5a8b3af455429971feec467b", "5a77d12755429967ab1052b8", "5ae66d515542992ae0d16346", "5ae12f16554299422ee9961f", "5ab8e70b5542991b5579f031", "5a7dab795542990b8f503a22", "5addd82c5542990dbb2f7eb9", "5ae5e852554299546bf82fb8", "5ab1ea0a554299340b52542a", "5abd2eb255429933744ab750", "5a875154554299211dda2be7", "5a7d26fd554299452d57bb2a", "5a714b645542994082a3e78b", "5a7cff5755429907fabef09d", "5adfc50355429942ec259b43", "5a8ac0595542992d82986f6c", "5a84f7255542991dd0999e33", "5a8a2a5555429930ff3c0ccc", "5abbe40555429931dba145cd", "5ae446fc5542995ad6573d24", "5a7daad55542995f4f4022b7", "5a83587f5542990548d0b1de", "5ae807d25542993210984045", "5a7252db5542990c210a4104", "5abffd10554299012d1db556", "5a8c7595554299585d9e36b6", "5a7b2c9155429927d897bf51", "5a75e29f5542992d0ec0600c", "5a84b4005542991dd0999d99", "5ae47ae05542995ad6573d4f", "5adde5155542997dc7907092", "5abaf3195542996cc5e49f2c", "5a9052765542995651fb513d", "5ab2979c554299449642c919", "5add3e2b5542997545bbbcf5", "5a80be9655429938b61421b5", "5abf16965542993fe9a41dd3", "5abd0553554299700f9d7963", "5ac3b4cc554299741d48a2eb", "5ae3639b5542991a06ce99b2", "5ab30336554299166977417c", "5a807181554299485f5985ec", "5a76139b5542994ccc9186be", "5a76a0005542993569682c64", "5a77bef05542992a6e59dfaf", "5a7c220b5542990527d5545c", "5ac285335542996773102552", "5adbf3fa554299438c868ce0", "5ab21682554299722f9b4ca9", "5a88341c5542994846c1ce20", "5a90719b5542995b442420b7", "5ae17daa55429901ffe4aea9", "5a8f54165542992414482a52", "5ae3fd2c5542995dadf2428f", "5a7b22f8554299042af8f6ff", "5ab235ed5542993be8fa98c8", "5a8900c75542997e5c09a6ed", "5a7d5c175542991319bc93b1", "5ac29f58554299657fa28fe4", "5adccdab5542994d58a2f6db", "5a82119155429926c1cdae28", "5ae0613d5542993d6555eb22", "5a8d0c74554299585d9e37a1", "5ab28019554299722f9b4d51", "5adefe645542992d7e9f926c", "5adfcc625542992d7e9f93d3", "5aba7c7155429955dce3ee58", "5ab66fd355429954757d32b7", "5adbd1d25542994650320bf6", "5a8f5a185542992414482a71", "5a7a3a945542996a35c17147", "5a808afb5542992bc0c4a76b", "5ac02880554299012d1db5c0", "5abdb5b455429965af743dfa", "5a7c25895542997c3ec972e6", "5a7e04b05542995f4f40238a", "5a73b0975542992d56e7e387", "5ae1e22a5542997f29b3c160", "5ac258cc554299636651998e", "5ae1cf355542997f29b3c126", "5adf8dea5542995534e8c7e4", "5ae74818554299572ea547ad", "5ab6993f554299110f219a2b", "5a84e1095542997175ce1f25", "5a77c15f5542997042120b1c", "5ab5d43f5542997d4ad1f1f6", "5a8d8f77554299068b959d3e", "5ab3f6b9554299753aec5a03", "5a8494b15542991dd0999d35", "5a86f2fb554299211dda2b64", "5ab7b3cc55429928e1fe3893", "5ab43b755542991779162c21", "5a7cc58f554299452d57ba45", "5a904b0c5542995651fb510e", "5ae088525542993d6555eba1", "5ab5f372554299488d4d9a60", "5a73630655429901807db020", "5ac046c65542996f0d89cb71", "5adcb7eb5542994ed6169bd0", "5ae0aefe554299603e418437", "5a88e7a55542993b751ca8ca", "5ae184185542990adbacf7ac", "5a89fea655429970aeb701eb", "5a89ba6b554299669944a584", "5a838eba5542992ef85e22fe", "5ab61668554299488d4d9aa6", "5ac3b62b554299741d48a2f6", "5ab7452b5542992aa3b8c806", "5a8b69f45542997f31a41d19", "5a864ff7554299211dda2af2", "5ab1d7ac554299449642c7e6", "5a750ccc5542993748c897a8", "5adc760c5542994650320d11", "5abd93115542996e802b47e0", "5a89da93554299669944a5cc", "5ae55d1e55429960a22e02cb", "5abbe09d55429931dba145c0", "5ac43e0f554299076e296d8e", "5a8765ad5542996e4f3087cf", "5ae142c455429920d5234355", "5abba4b4554299642a094af3", "5a85f8665542994775f606ce", "5ade28b15542997c77aded83", "5ab8ee805542991b5579f051", "5a73bcf055429978a71e909c", "5a7e075c5542995f4f4023a3", "5a75383a5542996c70cfaeb3", "5ae226bd5542994d89d5b379", "5ae1f79f5542997283cd2302", "5add2397554299473435386f", "5a838fcd5542992ef85e2305", "5a84973e5542992a431d1a67", "5a8dae82554299068b959d99", "5abd96fe5542992ac4f382d9", "5a8a484f5542996c9b8d5e94", "5a8c78a8554299585d9e36bc", "5ab3e45655429976abd1bcd4", "5ac4d717554299076e296e0e", "5a84e0a45542991dd0999e11", "5a74655b55429974ef308bf0", "5a7a7e9f5542994f819ef1f4", "5ab95a63554299743d22eb06", "5a79118755429974737f7962", "5a89a82c5542993b751ca973", "5a90997b5542995651fb51af", "5ade881f5542992fa25da7af", "5a8a12ae5542992d82986e92", "5ab6654455429954757d3281", "5ab3952e5542992ade7c6de8", "5a7782b95542995d8318117f", "5adec96e5542995534e8c712", "5ab480c45542990594ba9c36", "5a7a567255429941d65f25bd", "5ac3a0b25542993915413838", "5ac259585542992f1f2b3868", "5a8c4ebf554299585d9e365d", "5a807ea1554299485f598624", "5ab997ad554299131ca4238c", "5a72a5a75542992359bc314e", "5ac31ad4554299741d48a1ee", "5a73b8d655429908901be31f", "5a7c4a6455429907fabeef53", "5ab7de455542992aa3b8c864", "5abfa9745542997ec76fd426", "5a71474b5542994082a3e74f", "5a88696b554299206df2b25b", "5a7f95685542995d8a8ddeaa", "5ae4c01e55429913cc2044f3", "5ae4c49a5542995dadf24400", "5add92885542990dbb2f7e72", "5ab3244c554299233954ff29", "5a8df72055429917b4a5bcf2", "5abdd0f15542991f6610604d", "5a7a5aea5542996c55b2dd6f", "5adc2e3d554299438c868d6b", "5abff0785542997d64295957", "5abbb116554299642a094b1d", "5a8b97485542995d1e6f140f", "5a8ef1e35542995a26add57a", "5a7222b155429971e9dc92c4", "5ab6e07b55429954757d33a0", "5ae16b925542997b2ef7d1e7", "5addfdf65542992200553bd2", "5a7284c85542991f9a20c4c0", "5ab66b63554299110f219a0a", "5abcfc365542993a06baf9d6", "5ac120c55542992a796dedfc", "5ae7fd085542993210984007", "5ae28c0a554299495565daaa", "5addd41b5542990dbb2f7e94", "5ac26eed55429951e9e685bf", "5ae362105542991a06ce99ae", "5ae35b035542992e3233c3ae", "5abb76fa5542992ccd8e7f48", "5a7c465a5542990527d5547f", "5abb73425542996cc5e49ff5", "5a79b495554299148911fa3b", "5a802bf05542992bc0c4a6b7", "5a85cead5542991dd0999ea9", "5ac3d10b5542993915413952", "5ae61351554299546bf83066", "5a8e2fe95542995085b373d0", "5abdcad155429965af743e4d", "5a89fa2655429970aeb701d2", "5ac29a835542990b17b1542f", "5ab53ef1554299488d4d9920", "5abcf14f55429959677d6b5b", "5a7efd2f5542994959419a9c", "5ae791245542997b22f6a7a3", "5adfecd755429942ec259b90", "5ae54eea5542993aec5ec19d", "5a7b8cc7554299294a54a9ea", "5a7c572655429935c91b5166", "5adcd1665542994d58a2f6e6", "5a7e71ab55429949594199bc", "5a8b84905542997f31a41d5d", "5a8671e95542994775f60763", "5abea7c25542990832d3a068", "5ae3ff2d5542995dadf242a9", "5a7dd8cb5542995f4f4022d0", "5a8f73455542997ba9cb3269", "5a8e68b15542995085b3742d", "5adf88995542993344016cd4", "5ae0536755429924de1b70a6", "5ac434ed554299204fd21f09", "5ab9374b554299131ca422c6", "5abe79c755429976d4830b2a", "5a8ba07d5542996e8ac88985", "5a8e36835542995085b373e2", "5adfda9655429925eb1afac0", "5abf63f15542997ec76fd3ea", "5ab472a35542991779162c98", "5a8fab8c5542995b4424208a", "5a8adc935542992d82986fb0", "5abbe4c0554299642a094bc0", "5ac5275755429924173fb617", "5ab3b42c5542992ade7c6e4f", "5ab3dbd8554299233954ffd6", "5abe54585542991f66106148", "5a8df1a65542995085b3735f", "5a778d7855429967ab1051fa", "5ac2546a554299636651997b", "5a75441c55429916b01642cf", "5ac3c10a554299204fd21e33", "5a75e16055429976ec32bc6a", "5abb8e2d554299642a094aa4", "5a8cd291554299441c6b9f14", "5a776ac75542993569682d9b", "5a77beae5542992a6e59dfae", "5a7c2819554299683c1c62db", "5ac2a35055429967731025ce", "5a89706055429951533612f9", "5ac4a7615542996feb3fe92a", "5abd516a5542992ac4f3825c", "5adf51595542993344016c52", "5a7eb22655429930675135cb", "5adcfba05542990d50227d83", "5a8c7623554299653c1aa07a", "5a85a3005542991dd0999e72", "5a836a2d554299334474600f", "5a83d42d5542992ef85e2375", "5a891e1e5542993b751ca913", "5ac273455542992f1f2b38d4", "5ab6cd0e554299710c8d1f68", "5a849c1b5542997175ce1ef9", "5a77691455429966f1a36d2b", "5a7147fd5542994082a3e758", "5ae7525f5542991e8301cc60", "5a7f6f4b5542994857a76740", "5a7526945542993748c897d2", "5a738b855542992d56e7e363", "5a89e98955429946c8d6ea0c", "5a85b2ac5542991dd0999e90", "5a7746cf5542994aec3b7263", "5abd583255429924427fcfb9", "5a78d9c455429970f5fffda7", "5ae6479a55429929b0807b1b", "5abbd12b55429931dba14577", "5a7f3bcd55429930675136c3", "5adfa12c554299025d62a301", "5ac532335542994611c8b41c", "5ae222ec5542996483e648ff", "5a75eca555429976ec32bca8", "5ae1465855429920d5234373", "5ae8161555429952e35eaa04", "5a80abd15542996402f6a5e0", "5a8f9dca55429918e830d295", "5a7c6e435542996dd594b933", "5ae0575755429945ae95931f", "5ade6fcd55429975fa854ecf", "5ab2d1c45542992953946877", "5a845b2d5542996488c2e520", "5a775f615542993735360250", "5ab7f0b9554299366779405d", "5a7f10e2554299306751360b", "5ab9fe1255429939ce03dc40", "5ae25d2b554299495565da46", "5ab90f5755429934fafe6e57", "5a8ed8b955429917b4a5bdd3", "5ab3e577554299233954ffe3", "5ade04465542990dbb2f7f2f", "5a7e05635542995f4f40238f", "5a7943c655429970f5fffe89", "5a862b0b554299211dda2a98", "5a77b6535542995d83181274", "5ab67c3f55429954757d32f9", "5ae5559c55429960a22e02ba", "5ab72a035542993667793f5c", "5ac0d981554299012d1db646", "5adec4695542992fa25da83a", "5abae88c554299660624167e", "5ae765185542995703ce8c0f", "5a881d2355429938390d3eeb", "5a7636fb55429976ec32bd79", "5adc6825554299438c868da1", "5ab6fcf55542991d32223718", "5a8e4d9855429917b4a5bd8d", "5ab40e1c5542996a3a969f00", "5ab96ecb5542996be20204e6", "5a88451655429938390d3f12", "5a7ddc5a5542995f4f4022da", "5a7b89a3554299294a54a9e3", "5ae798ce55429952e35ea995", "5ab8284d5542990e739ec83f", "5ae33f9a5542990afbd1e0f1", "5a79fc8b5542994f819ef114", "5a906a685542990a9849362e", "5a86038455429960ec39b600", "5ae5be02554299546bf82f3e", "5a7d68e25542995f4f402270", "5a88fa1c5542995153361219", "5a8468c95542990548d0b2b1", "5ae414d855429970de88d8ec", "5a90bd7b55429916514e7547", "5aba77be55429955dce3ee44", "5adcaa7f5542990d50227cdb", "5ae4585c5542996836b02c89", "5adcde255542992c1e3a243f", "5a8ad8b65542992d82986fa2", "5ab82bbb5542990e739ec84c", "5ab808ea5542991d3222380e", "5ab5fddb554299110f2199af", "5ac1ff2255429964131be2b3", "5a7cba005542990527d55516", "5a8da0b85542994ba4e3dcd3", "5a7e08305542995f4f4023ab", "5ab73b2e5542993667793f9b", "5a76c98855429972597f13d0", "5ab9560855429970cfb8ea73", "5a8ac38855429970aeb70300", "5a8bd4d65542997f31a41dd8", "5ab8883455429916710eb084", "5ab48f245542990594ba9c66", "5a81d19b55429926c1cdad90", "5ab44a775542991779162c57", "5a80427c5542996402f6a49f", "5aba549f554299232ef4a290", "5ab959905542996be2020497", "5a869e845542991e7718166f", "5ab42b8955429942dd415ea6", "5a823b0c55429940e5e1a804", "5ac537975542996feb3fea3c", "5ade64b655429939a52fe89a", "5ae63e3255429929b0807aff", "5ab52bd1554299637185c4ee", "5ae4accf5542995ad6573e19", "5ab6a2be554299710c8d1f08", "5ae1e9fd5542997f29b3c1a1", "5a792f3e554299029c4b5f20", "5a7f732d5542992e7d278d02", "5ae0c7e755429945ae95944c", "5a738093554299623ed4abbb", "5a7d61645542997cc2c47455", "5a8c5c6f554299653c1aa04d", "5a7136e25542994082a3e67e", "5a8af56855429950cd6afc2f", "5a85eed75542996432c5713b", "5a77a3a355429967ab105211", "5ac3295d5542995ef918c14d", "5a8a521d5542992d82986f15", "5ac2583b55429951e9e68565", "5a8b7e9c5542995d1e6f13c1", "5ac3d9c9554299204fd21ea3", "5a78c66e55429974737f787b", "5a7a633a55429941d65f2609", "5abefa5b5542997ec76fd3a3", "5ae0d5335542990adbacf674", "5ae277575542996483e649b2", "5ab25004554299722f9b4cd7", "5ade8802554299728e26c72d", "5ae690b055429908198fa622", "5a72953b5542994cef4bc357", "5a8cac04554299240d9c21ce", "5a74bc1555429929fddd84dd", "5a734a205542992359bc32e2", "5ab2fccb5542991669774176", "5ae20cd25542997283cd2376", "5adc6ded55429947ff17395d", "5a7600275542994ccc91867a", "5adf8f675542995534e8c7eb", "5ae731c4554299572ea54781", "5abaa8b355429901930fa87f", "5a80ac53554299485f5986a0", "5a72d3525542991f9a20c5ab", "5ab484415542990594ba9c44", "5a7133565542994082a3e65c", "5a80071f5542992bc0c4a684", "5a713a5a5542994082a3e6a9", "5ab965bb554299743d22eb56", "5adf5fa65542993a75d264db", "5a773d8955429966f1a36cc4", "5a8b20b85542996c9b8d5fb5", "5a7d043755429909bec768f7", "5ac248fd55429951e9e68525", "5a81b4cc5542995ce29dcc3c", "5ae70ed7554299572ea546a1", "5ae4b8fc55429913cc2044e5", "5ae0036a55429942ec259bdf", "5adc294c5542996e685252cf", "5a7df1095542990b8f503afe", "5add64065542995b365fab25", "5a7ab1e955429931da12c95d", "5ab3010455429976abd1bc15", "5a841b215542990548d0b25b", "5ae4818f5542995ad6573d5b", "5a8511185542992a431d1aca", "5add48465542992200553a80", "5a7a45425542990783324ee2", "5abdd8d95542991f6610606a", "5add44f35542997545bbbd0c", "5ae095f055429924de1b7103", "5a7bf0d75542996dd594b85c", "5a7dbb605542997cc2c47486", "5a77671255429966f1a36d21", "5ac199955542994ab5c67d8c", "5ab32fc355429969a97a80fa", "5abe822155429976d4830b48", "5a8cfee555429941ae14df5c", "5ae200655542994d89d5b2f4", "5a82efe355429966c78a6aa1", "5ab266b5554299340b5254b4", "5ae4ca6355429913cc204507", "5ae707a25542991bbc9761a7", "5ab7f1315542991d322237d1", "5a7b63eb55429931da12ca7e", "5ab2a54b554299295394678e", "5abd6bf055429933744ab7c8", "5ab9d44255429939ce03dc36", "5ae5a8435542992663a4f208", "5a8eb63455429917b4a5bdb7", "5a7c54eb554299683c1c62e8", "5abd2cde5542993062266bb1", "5a77c1d95542995d831812a6", "5ae6e36155429908198fa6d3", "5a84c4bb5542991dd0999dec", "5a8ae00e5542992d82986fc7", "5adf4a275542993a75d26498", "5abd889f55429924427fd046", "5a776fc15542997042120a3a", "5a7d7d895542995f4f402281", "5ae0ee2c5542993d6555eca3", "5ae0361155429925eb1afc2c", "5a8230af554299676cceb227", "5a904f6355429916514e746e", "5a8ad2f3554299515336138d", "5a7a3dbb5542996a35c17157", "5a84d2775542994c784dda5f", "5a8ee4315542990e94052ba7", "5a79eef75542996c55b2dca6", "5a77f4755542992a6e59e01f", "5ae0f97a5542997b2ef7d0c1", "5a8db19d5542994ba4e3dd00", "5adf86b45542993344016cc1", "5ae604ec5542996de7b71af9", "5a78f753554299078472776e", "5ab5cdb85542997d4ad1f1e0", "5abea9a35542990832d3a07e", "5abbd3b4554299642a094b4f", "5ae2138c5542997283cd23b7", "5a79c9cc554299148911fa53", "5a7613c15542994ccc9186bf", "5ac23f5255429951e9e684d8", "5ab2d3df554299194fa9352c", "5a7d9bc65542990b8f5039cb", "5adf07f25542992d7e9f9279", "5ab28156554299449642c8ce", "5abd6cb955429933744ab7cd", "5a898872554299669944a540", "5a8ceb7a554299441c6b9f7b", "5ab3e4ee554299233954ffe1", "5a7dd9445542997cc2c474ac", "5a7323ad5542994cef4bc476", "5ae180d855429901ffe4aecc", "5a7cc5f9554299452d57ba49", "5adcc6765542994d58a2f6c9", "5ab412da5542996a3a969f15", "5ab225565542993be8fa98bf", "5adcc7515542994ed6169bff", "5ab84bab55429934fafe6d49", "5adffd4755429906c02daa9b", "5a8675b1554299211dda2b44", "5abcf84d55429959677d6b86", "5a7131565542994082a3e64f", "5ade9c9355429975fa854f1b", "5ab850b255429916710eb036", "5ae8242455429952e35eaa54", "5ab85b1755429916710eb041", "5ae29d8a5542996483e64a0f", "5a7b8d4955429927d897bfeb", "5ab4fd0b55429942dd415fbb", "5ae1b01b5542997283cd2237", "5a733caa5542992359bc32a3", "5ab7178f5542991d32223774", "5ab3b50055429969a97a818c", "5adc459255429944faac24ad", "5a8c66db554299585d9e3683", "5abe49765542991f6610611f", "5ac0cfb45542997d64295a58", "5a72bbfa5542992359bc318d", "5a886211554299206df2b24a", "5ae22895554299495565da0c", "5ab568af554299494045efaf", "5a77e96c5542995d8318132d", "5a8cea40554299441c6b9f76", "5a7b55bc55429927d897bf98", "5abd957a5542996e802b47eb", "5ae532e955429908b632656f", "5ab6973f554299110f219a25", "5a7a098d5542996c55b2dce3", "5a7125165542994082a3e5d0", "5adf3feb5542993a75d2644c", "5ae18ae6554299234fd04289", "5ac1c6805542994ab5c67e11", "5a8307c055429966c78a6af9", "5ac4e9ab5542996feb3fe974", "5ae7ad385542993210983ef0", "5ae5d008554299546bf82f7d", "5ab811395542990e739ec80f", "5ae0bf3a55429906c02dab21", "5ae0b68e55429906c02daafb", "5a866c04554299211dda2b25", "5ab3d69255429969a97a81c9", "5ab446d455429942dd415f07", "5ae346365542991a06ce9985", "5ae0c1be5542993d6555ec42", "5adfcb5455429906c02daa48", "5abea05f5542991f661061b6", "5a74b66a55429929fddd84cb", "5abced075542993a06baf9ac", "5ae1847e55429920d52343ee", "5ab3306a55429969a97a80fd", "5ab91e3255429916710eb117", "5ae52c0f5542992663a4f12e", "5ae6638555429908198fa5ba", "5ac2a10e554299657fa28fec", "5adc3aff5542996e685252eb", "5a7324205542994cef4bc478", "5a80e03e55429938b614222c", "5abe3f3455429976d4830aa8", "5ac23ff0554299636651994d", "5ae121315542997b2ef7d0ee", "5abb07e95542992ccd8e7ec4", "5ae31bbd55429928c42395fc", "5adcca7f5542994ed6169c06", "5ab9cfe655429970cfb8ebaf", "5ade48795542992fa25da729", "5ae135fb55429920d523431f", "5a7a7dc15542994f819ef1f2", "5a7732dc55429972597f149b", "5a8b4a5f55429949d91db531", "5ab42ae95542991751b4d6c4", "5a8331f15542993344745fe8", "5ac2a20055429967731025cb", "5a7f010955429934daa2fc8b", "5a8637415542994775f60724", "5abaf6865542996cc5e49f3c", "5ae2b794554299495565db10", "5a82bdd655429940e5e1a92a", "5ab2f53d5542991669774157", "5ae6196d5542995703ce8afd", "5adbfc945542996e68525296", "5ae37ec25542990afbd1e178", "5adcea3755429947343537ca", "5abe364e5542993f32c2a08e", "5a7db1a15542995ed0d16660", "5a85ae455542997175ce1ff0", "5a84aca95542992a431d1a89", "5a89bb0b5542992e4fca83a0", "5a79395455429970f5fffe7d", "5a7f3c8f55429930675136c9", "5a8d02cc554299441c6b9fb6", "5a74d64055429916b0164223", "5ac2e9c45542990b17b154a0", "5ab724b45542991d32223785", "5adfb698554299603e418385", "5ae28b76554299495565daa8", "5ac116595542992a796dedd1", "5a72af155542992359bc316c", "5a8777395542996e4f3087f3", "5adbfb6b55429947ff17388d", "5ac290e85542992f1f2b3916", "5a84ad3a5542992a431d1a8b", "5a7f74975542992097ad2f77", "5ab69f13554299710c8d1ef3", "5ab6c63755429953192ad38b", "5ab615cf554299488d4d9aa3", "5a8ba92c554299240d9c2071", "5ae7eaf65542994a481bbe1d", "5abb2f2f554299232ef4a3d7", "5a8ba59a554299240d9c2058", "5a78c56455429974737f7876", "5ae796af554299540e5a5623", "5aba9813554299232ef4a357", "5ae1e6d95542997283cd22c3", "5a85a37d5542997175ce1fe5", "5a8769475542993e715abf2b", "5a863b87554299211dda2ab7", "5a89dc81554299669944a5dc", "5a7b215f55429931da12c9e3", "5a73d33e5542992d56e7e3a9", "5a7d115355429909bec76934", "5a7b681b554299042af8f767", "5add4fd15542992200553a98", "5a84c89a5542992a431d1a9c", "5a7fa0d55542995d8a8dded3", "5a8333115542993344745feb", "5a7624b85542992db947372c", "5aba7a22554299232ef4a2e7", "5abf12f25542997719eab667", "5ae07db855429945ae95937a", "5a88d745554299206df2b378", "5a7a01b15542996c55b2dcd1", "5a8fa4d95542995b4424207b", "5adc8d4f554299473435373d", "5a8e06785542995085b37383", "5a8ab6be5542992d82986f5f", "5a78d16455429974737f78bb", "5adc368155429944faac2468", "5ae5d63b5542996de7b71a2d", "5ae327e05542992f92d82220", "5ac4e476554299076e296e34", "5a763986554299109176e6bd", "5a7a05485542990783324def", "5a75e67f55429976ec32bc8b", "5a8b2d3255429950cd6afc60", "5abff0625542997ec76fd476", "5abd420f55429933744ab75c", "5ae3790f5542992e3233c415", "5a88c4175542997e5c09a6ae", "5ae202f2554299234fd0438d", "5adfa97555429942ec259ae1", "5a8220db5542995ce29dccc1", "5abadc355542996606241644", "5a71095e5542994082a3e4f3", "5abc817c5542993a06baf8fd", "5a8a547d55429970aeb70293", "5a837121554299123d8c2128", "5ab8680755429934fafe6da8", "5a78abbc554299148911f90c", "5ab41d855542996a3a969f4b", "5a7b43735542992d025e67cb", "5ac2610f55429951e9e68593", "5ab9788155429970cfb8eb1a", "5a75377c55429929fddd8514", "5a7f5e0455429969796c1a12", "5a8f54215542992414482a54", "5a77414155429972597f14d7", "5abbdd565542993f40c73bfb", "5a72830e5542994cef4bc2f9", "5ac4f8705542995c82c4ad8d", "5ab26a4b55429970612095fc", "5a7b3b4f5542992d025e67ac", "5ae648b955429929b0807b1d", "5ab9b1fd5542996be202058e", "5aba92da55429901930fa840", "5a75e05c55429976ec32bc5f", "5a7a6ec05542990783324f49", "5ac13d5c5542996f0d89cc78", "5a8a502b5542996c9b8d5ea6", "5a8b98665542995d1e6f1413", "5a83e98e5542992ef85e2387", "5a7e96f655429934daa2fc39", "5ae738f75542991bbc9761c4", "5adfb78f5542995ec70e9081", "5ae45b6b55429970de88d943", "5abd0097554299700f9d7956", "5abff65c5542994516f45551", "5ac2b8085542990b17b1546b", "5a7ce50f55429909bec76869", "5ab93781554299131ca422c7", "5a82ffef55429954d2e2ebf2", "5a75d45f5542994ccc918630", "5a848a685542997b5ce3feae", "5ae0bb2c5542993d6555ec27", "5ab8ed305542991b5579f046", "5ae4ec1e5542990ba0bbb19b", "5ab946d7554299743d22eaaf", "5a76394c5542994ccc918725", "5a7fa08d55429969796c1af7", "5a7df7115542990b8f503b0f", "5a89007355429938390d4026", "5a8c62f5554299585d9e367b", "5a8359c7554299123d8c20fb", "5ae6179b5542992663a4f25b", "5ab5f9a0554299488d4d9a6d", "5abc0ac455429959677d6a01", "5ab98d70554299753720f82f", "5ab678f455429954757d32d3", "5a88940055429938390d3f7b", "5ab852c05542990e739ec8c5", "5a7f354b5542992e7d278c8c", "5a7a03225542996c55b2dcd5", "5abce87e5542993a06baf9a5", "5ae61e7555429929b0807adf", "5ab6644655429954757d327d", "5a8ee2855542995a26add550", "5a7aa2a255429941d65f270e", "5abf10b75542997719eab659", "5a7fa97d5542994857a767a9", "5a8ba45b554299240d9c2051", "5a7eec6f55429934daa2fc7c", "5a7c7c225542996dd594b93f", "5ae0198455429942ec259c1e", "5ae0132d55429925eb1afc00", "5add2b865542992c1e3a2550", "5a8dce155542994ba4e3dd24", "5a8b2ccf55429971feec4665", "5ab6f0c25542995eadef00ea", "5a7c988755429935c91b524d", "5ac2f1d1554299218029dba4", "5ae01fcf554299025d62a411", "5ae3ffba5542995dadf242b2", "5a7fe7f85542994857a7683e", "5adc020a5542994650320c4c", "5ac009bf5542997d642959a7", "5ab9535a5542996be202047e", "5add7ce55542992ae4cec585", "5a87bd4e5542996432c57279", "5ac02d59554299294b218f3b", "5aba55f25542994dbf0198e0", "5a875b755542996e4f308796", "5ae446175542995ad6573d1f", "5a7698c2554299373536010d", "5a770bb855429937353601b5", "5ae0847455429945ae959394", "5a90cb205542995651fb51fe", "5ade7feb5542992fa25da799", "5abc553a554299700f9d7871", "5ab1f13e554299449642c81c", "5abc7bf75542993a06baf8f0", "5a85db6e5542994c784ddb96", "5a779be75542995d83181215", "5a83f07455429933447460be", "5a72df275542992359bc31b3", "5a8484495542997175ce1ee8", "5a735c3a55429901807daff7", "5a835b88554299123d8c2101", "5a85e2775542994775f60680", "5a8c03415542995e66a4752a", "5a88a4e65542997e5c09a671", "5a74524455429979e28828f6", "5aba6d0f55429955dce3ee19", "5a8f03855542997ba9cb3186", "5ac1c18855429964131be279", "5ab804305542995dae37ea14", "5ac3e80b554299076e296ccd", "5ab871725542990e739ec8f8", "5ac4edf155429924173fb4e3", "5ae164685542997b2ef7d1cb", "5adcc90c5542990d50227d1b", "5a734f4455429901807dafba", "5a9084e75542990a98493667", "5a7a073a5542990783324dfb", "5a7621bb554299109176e650", "5ac0d5a1554299294b219025", "5abe36745542991f66106101", "5a7b3f4d5542992d025e67bb", "5a8dacf2554299068b959d91", "5a8463945542992ef85e23d9", "5ae8202255429952e35eaa3a", "5a88abf75542997e5c09a6a1", "5a7de14a5542991319bc93fc", "5ae5f47c5542996de7b71a82", "5a74760b55429974ef308c03", "5a7780e855429949eeb29e9f", "5ac279345542990b17b153b0", "5a87bd4e5542994846c1cde0", "5ab82dbb5542990e739ec857", "5a74d9c555429929fddd84fb", "5a75d9b4554299109176e5a7", "5ac54ca45542993e66e822d5", "5add7c9c5542990dbb2f7e60", "5a8ff83c55429916514e73f3", "5ae21154554299495565d9d4", "5ae688065542996d980e7beb", "5abdf68c5542993f32c2a06b", "5ab3d2b7554299233954ffb8", "5ab576dc5542997d4ad1f13d"]}

## A0: zero-shot (off-the-shelf ms-marco MiniLM-L6, 22.7M params)

In [ ]:
!python score.py --model zeroshot

## A2: data-matched baselines (N = 50 / 200 train questions)
These are the fair baselines for the LLM weight-editing arm (exp03).

In [ ]:
!python train.py --n-train 50 --epochs 20 --tag n50 && python score.py --model n50

In [ ]:
!python train.py --n-train 200 --epochs 10 --tag n200 && python score.py --model n200

## A1: full fine-tune (~90k questions, ~900k pairs, 1 epoch)
Resumes from the Drive checkpoint if the runtime disconnected.

In [ ]:
!python train.py --n-train 0 --epochs 1 --tag full && python score.py --model full

## Analysis

In [ ]:
!python analyze.py

## Download the small result files (scores, summaries, train args; models stay on Drive)

In [ ]:
import shutil, glob, os
from google.colab import files
R = os.environ['RESULTS_DIR']
pack = '/content/exp02_results_small'
shutil.rmtree(pack, ignore_errors=True); os.makedirs(pack)
for f in glob.glob(f'{R}/*.json') + glob.glob(f'{R}/*.csv') + glob.glob(f'{R}/*/train_args.json'):
    dst = os.path.join(pack, os.path.relpath(f, R)); os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f, dst)
files.download(shutil.make_archive('/content/exp02_results', 'zip', pack))